# 02 — Model Training & Evaluation

This notebook trains the XGBoost model interactively and shows detailed evaluation charts.
Run cells one by one (Shift+Enter).

In [ ]:
import os, sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import yaml
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_curve, auc
)

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

pair      = cfg['pair'].replace('=X','').lower()
timeframe = cfg['timeframe']
print(f'Config loaded. Pair={pair} | Timeframe={timeframe}')

## 1. Load Train & Test Data

In [ ]:
label_dir  = f'../data/labelled/'
train_df   = pd.read_csv(f'{label_dir}{pair}_{timeframe}_train.csv', parse_dates=['datetime'])
test_df    = pd.read_csv(f'{label_dir}{pair}_{timeframe}_test.csv',  parse_dates=['datetime'])

NON_FEAT   = ['datetime','target','open','high','low','close','volume']
feat_cols  = [c for c in train_df.columns if c not in NON_FEAT]

X_train    = train_df[feat_cols].values
y_train    = train_df['target'].values
X_test     = test_df[feat_cols].values
y_test     = test_df['target'].values

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Features: {len(feat_cols)}')

## 2. Scale Features

In [ ]:
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)   # IMPORTANT: use fit from train only
print('Features scaled.')

## 3. Train Model

In [ ]:
m = cfg['model']
model = XGBClassifier(
    n_estimators     = m['n_estimators'],
    max_depth        = m['max_depth'],
    learning_rate    = m['learning_rate'],
    subsample        = m['subsample'],
    colsample_bytree = m['colsample_bytree'],
    random_state     = cfg['random_seed'],
    eval_metric      = 'logloss',
    n_jobs           = -1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)
print('\nTraining complete!')

## 4. Evaluate on Test Set

In [ ]:
proba  = model.predict_proba(X_test)[:, 1]
y_pred = (proba >= 0.5).astype(int)

print('Classification Report (all predictions):')
print(classification_report(y_test, y_pred, target_names=['DOWN','UP']))

# With confidence threshold
threshold = cfg['confidence_threshold']
confident_mask = (proba >= threshold) | (proba <= 1 - threshold)
print(f'\nWith {threshold:.0%} confidence threshold:')
print(f'  Trades taken: {confident_mask.sum()} / {len(y_test)} ({confident_mask.mean():.1%})')
if confident_mask.sum() > 0:
    y_conf = (proba[confident_mask] >= 0.5).astype(int)
    acc_conf = accuracy_score(y_test[confident_mask], y_conf)
    print(f'  Accuracy on confident rows: {acc_conf:.3f}')

## 5. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['DOWN','UP'], yticklabels=['DOWN','UP'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()

## 6. ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, proba)
roc_auc     = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='#4a90d9', label=f'AUC = {roc_auc:.3f}')
plt.plot([0,1],[0,1], 'k--', linewidth=0.8)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve  (AUC > 0.55 is useful)')
plt.legend()
plt.tight_layout()
plt.show()
print(f'AUC: {roc_auc:.3f}   (random = 0.50, perfect = 1.00)')

## 7. Feature Importance

Which features does the model rely on most?

In [ ]:
importance = pd.Series(model.feature_importances_, index=feat_cols)
top20 = importance.nlargest(20)

fig, ax = plt.subplots(figsize=(7, 6))
top20.sort_values().plot(kind='barh', ax=ax, color='#4a90d9')
ax.set_title('Top 20 Feature Importances')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## 8. Save Model

Saves to `models/` so `predict.py` and `backtest.py` can use it.

In [ ]:
os.makedirs('../models', exist_ok=True)
joblib.dump(model,     f'../models/{pair}_xgboost.pkl')
joblib.dump(scaler,    f'../models/{pair}_scaler.pkl')
joblib.dump(feat_cols, f'../models/{pair}_feature_cols.pkl')
print('Model, scaler, and feature columns saved to models/')

## 9. Quick Prediction Test

Manually test the model on the last row of test data.

In [ ]:
last_row   = test_df.iloc[[-1]]
X_last     = scaler.transform(last_row[feat_cols].values)
prob_up    = model.predict_proba(X_last)[0][1]
prob_down  = 1 - prob_up
threshold  = cfg['confidence_threshold']

signal = 'BUY' if prob_up >= threshold else ('SELL' if prob_down >= threshold else 'FLAT')

print(f"Last candle: {last_row['datetime'].values[0]}")
print(f"Close:  {last_row['close'].values[0]:.5f}")
print(f"Prob UP: {prob_up:.1%}   Prob DOWN: {prob_down:.1%}")
print(f"Signal: >>> {signal} <<<")